# Demo — POST /ai/query (ISO 42001)

**Requisitos** (desde la raíz del repo)

1. **`.env`** — `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `OPENAI_API_KEY`.  
   Si en el shell tienes `export OPENAI_API_KEY=...`, haz `unset OPENAI_API_KEY` para que mande el valor del archivo.
2. **Seeds** (una vez):
   ```bash
   source .venv/bin/activate
   PYTHONPATH=. python scripts/seed_prompts.py
   PYTHONPATH=. python scripts/seed_documents.py
   ```
3. **API**:
   ```bash
   uvicorn app.main:app --reload
   ```
4. **Auth** — header `Authorization: Bearer dev-token` (incluido en la celda siguiente).

Ejecuta las celdas en orden. Solo se imprime **`response_text`** (sin `trace_id`, chunks ni metadata).

> **GitHub no ejecuta el notebook:** solo muestra el código y las **salidas guardadas** en el `.ipynb`. Si ves un error antiguo en github.com, corre de nuevo en local y luego *Clear All Outputs* antes de commit, o deja outputs limpios como en este repo.

In [ ]:
import os

import httpx

BASE_URL = os.getenv("AI_CORE_URL", "http://127.0.0.1:8000")
HEADERS = {
    "Authorization": "Bearer dev-token",
    "Content-Type": "application/json",
}
PAYLOAD_BASE = {
    "use_case": "chatbot",
    "include_global": True,
    "country": "costa-rica",
    "framework_filter": ["iso_42001"],
}
TIMEOUT = httpx.Timeout(120.0)


def require_api() -> None:
    """Fail fast with a clear message if uvicorn is not running."""
    try:
        r = httpx.get(f"{BASE_URL}/health", timeout=5.0)
        r.raise_for_status()
    except httpx.ConnectError as e:
        raise SystemExit(
            f"Connection refused — no hay API en {BASE_URL}.\n"
            "Abre otra terminal en la raíz del repo y ejecuta:\n"
            "  source .venv/bin/activate\n"
            "  uvicorn app.main:app --reload\n"
            "Deja esa terminal abierta y vuelve a ejecutar esta celda."
        ) from e


require_api()
print(f"API OK: {BASE_URL}/health →", httpx.get(f"{BASE_URL}/health", timeout=5.0).json())

In [ ]:
QUERIES = [
    "¿Qué es ISO/IEC 42001 y para qué sirve?",
    "¿Qué dice ISO 42001 sobre gestión de riesgos en sistemas de IA?",
    "¿Qué roles y responsabilidades de gobernanza menciona ISO 42001?",
    "¿Cómo aborda ISO 42001 el ciclo de vida del sistema de IA?",
    "¿Qué implica la mejora continua en un sistema de gestión de IA según ISO 42001?",
]

with httpx.Client(base_url=BASE_URL, headers=HEADERS, timeout=TIMEOUT) as client:
    for i, query in enumerate(QUERIES, 1):
        r = client.post("/ai/query", json={**PAYLOAD_BASE, "query": query})
        r.raise_for_status()
        print(f"\n--- Query {i} ---\n{query}\n")
        print(r.json()["response_text"])